In [0]:
%run "/Workspace/Users/jogesh.rajiyan@axahealth.co.uk/Utilities"

In [0]:
%sql
Create or replace temporary table MOL as
SELECT DISTINCT b.CaseNumber
,b.PolicyNumber
,a.CaseID
,b.subject
,b.CaseRequestType
,a.SecureMessageID
,e.IsCreatedByCustomer AS CustomerMessage
, e.AH_MessageContent__c
,case
      when e.iscreatedbycustomer = 'Yes' then 'Member'
      when e.iscreatedbycustomer = 'No' then 'Agent'
      else 'OTHER'
    end as MessageCreatedBy
,CONCAT(
      case
        when e.iscreatedbycustomer = 'Yes' then 'Customer'
        when e.iscreatedbycustomer = 'No' then 'Agent'
        else 'OTHER'
      end,
      ':',
      a.MessageTimestamp,
      e.AH_MessageContent__c
    ) as MessageContent,
    CONCAT_WS(
      '||',
      COLLECT_LIST(
        CONCAT(
          a.MessageTimestamp,
          ': ',
          CASE
            WHEN e.iscreatedbycustomer = 'Yes' THEN 'Customer'
            WHEN e.iscreatedbycustomer = 'No' THEN 'Agent'
            ELSE 'OTHER'
          END,
          ':',
          e.AH_MessageContent__c
        )
      ) OVER (
          PARTITION BY b.casenumber
          ORDER BY a.MessageTimestamp
          ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )
    ) AS ConversationContent
,CASE
      WHEN e.AH_MessageContent__c LIKE 'Your claim:%' THEN 'Agent'
      WHEN e.AH_MessageContent__c LIKE '%Using our online outpatients service%' THEN 'Agent'
      WHEN e.iscreatedbycustomer = 'Yes' THEN 'Customer'
      WHEN e.iscreatedbycustomer = 'No' THEN 'Agent'
      ELSE 'OTHER'
    END as MessageCreatedBy2
,CONCAT_WS(
      '||',
      COLLECT_LIST(
        CONCAT(
          a.MessageTimestamp,
          ': ',
          CASE
            WHEN e.AH_MessageContent__c LIKE 'Your claim:%' THEN 'Agent'
            WHEN e.AH_MessageContent__c LIKE '%Using our online outpatients service%' THEN 'Agent'
            WHEN e.iscreatedbycustomer = 'Yes' THEN 'Customer'
            WHEN e.iscreatedbycustomer = 'No' THEN 'Agent'
            ELSE 'OTHER'
          END,
          ': ',
          e.AH_MessageContent__c
        )
      ) OVER (
          PARTITION BY b.casenumber
          ORDER BY a.MessageTimestamp
          ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )
    ) AS ConversationContent2
,CONCAT_WS(
      '||',
      COLLECT_LIST(
        CONCAT(
          CASE
            WHEN e.AH_MessageContent__c LIKE 'Your claim:%' THEN 'Agent'
            WHEN e.AH_MessageContent__c LIKE '%Using our online outpatients service%' THEN 'Agent'
            WHEN e.iscreatedbycustomer = 'Yes' THEN 'Customer'
            WHEN e.iscreatedbycustomer = 'No' THEN 'Agent'
            ELSE 'OTHER'
          END,
          ':',
          a.MessageTimestamp,
          ': ',
          e.AH_MessageContent__c
        )
      ) OVER (
          PARTITION BY b.casenumber
          ORDER BY a.MessageTimestamp
          ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )
    ) AS ConversationContent3
,'MOL' AS ContactChannel
, a.MessageTimestamp
,a.ResponseTimestamp
, a.MessageSequenceNumberInCase
, a.CustomerMessageSequenceNumberInCase
, a.ResponseAgentID
, c.AgentName
,f.HasTransferFlag
,f.HasAgentResponseFlag
, SUM(f.HandleTimeSecs) AS HandleTimeSecs
,SUM(f.AgentResponseTimeSecs) AS wait_time
FROM axahealth_dataplatform_pr_gold.processed_layer.fact_secure_message_event a
LEFT JOIN axahealth_dataplatform_pr_gold.base_layer.secure_message_case b
ON a.SecureMessageCaseKey = b.SecureMessageCaseKey
LEFT JOIN axahealth_dataplatform_pr_gold.processed_layer.dim_worker_people c
      ON a.PeopleKey = c.PeopleKey
LEFT JOIN axahealth_dataplatform_pr_silver.salesforce.message__c e
ON a.SecureMessageID = e.Id
LEFT JOIN axahealth_dataplatform_pr_gold.processed_layer.fact_secure_message_case_agent_work f
ON b.SecureMessageCaseKey = f.SecureMessageCaseKey AND c.PeopleKey = f.PeopleKey
WHERE b.IsCreatedByCustomer = "Yes" AND MessageTimestamp > "2025-07-01"
GROUP BY ALL

In [0]:
%sql
select distinct
  CaseRequestType,
  count(DISTINCT casenumber)
from
  MOL
where
  MessageCreatedBy = 'Member'
Group by
  CaseRequestType

In [0]:
%sql
Create or replace temporary view MOLUniqueMessages as
select
  casenumber,
  policynumber,
  AH_MessageContent__c,
  min(MessageTimeStamp) as MinMessageTimestamp
from
  MOL
group by
  all;

select
  *
from
  MOLUniqueMessages
where
  casenumber = '11887956'

In [0]:
%sql
Create or replace temporary table MOLClean as
select
  a.*
from
  MOL a
    inner join MOLUniqueMessages b
      on a.casenumber = b.casenumber
      and a.AH_MessageContent__c = b.AH_MessageContent__c
      and a.MessageTimestamp = b.MinMessageTimestamp;

select
  *
from
  MOLClean
where
  casenumber = '11887956'

In [0]:
%sql
Create or replace temporary table MOLCustomerMessage as
select
CaseNumber
,policynumber
,SecureMessageId
,MessageTimestamp
,MessageSequenceNumberInCase
,AH_MessageContent__c
,CustomerMessageSequenceNumberInCase
,ContactChannel
,ResponseTimestamp
,ResponseAgentId
,AgentName
,HasTransferFlag
,HandleTimeSecs
,wait_time
, LAG(MessageTimestamp) OVER(Partition by casenumber ORDER BY MessageTimestamp,MessageSequenceNumberInCase) AS PreviousMessageTimestamp
, timestampdiff(second,
LAG(MessageTimestamp) OVER(Partition by casenumber ORDER BY MessageTimestamp,MessageSequenceNumberInCase),MessageTimestamp
) AS GapFromPreviousMessageSecs
from
  MOLClean
  WHERE MessageCreatedBy2 = 'Customer'
ORDER BY MessageTimestamp, MessageSequenceNumberInCase
;
SELECT * FROM MOLCustomerMessage WHERE casenumber = "11887956";

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW MOLCustomerAgentResponse AS 
SELECT c.casenumber
,c.policynumber
,c.SecureMessageId
,c.MessageTimestamp AS CustomerMessageTimestamp
,c.CustomerMessageSequenceNumberInCase
,c.AH_MessageContent__c AS CustomerMessage
,c.ResponseTimestamp
,a.SecureMessageId AS AgentSecureMessageId
,a.MessageTimestamp AS AgentMessageTimestamp
,a.MessageSequenceNumberInCase AS AgentMessageSequenceNumberInCase
,a.AH_MessageContent__c AS AgentMessage
,c.ResponseAgentId
,c.AgentName
,c.HasTransferFlag
,c.HandleTimeSecs
,c.wait_time
,c.ContactChannel
,c.PreviousMessageTimestamp
,c.GapFromPreviousMessageSecs
FROM MOLCustomerMessage c
LEFT JOIN MOLClean a
ON c.casenumber = a.CaseNumber
AND a.MessageCreatedBy2='Agent' AND a.MessageTimestamp = c.ResponseTimestamp
;
SELECT * FROM MOLCustomerAgentResponse WHERE casenumber = "11887956"

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE mol_to_claim AS
SELECT a.*
,b.ClaimNumber
,b.MembershipNumber
,MIN(a.CustomerMessageTimestamp) OVER(PARTITION BY a.casenumber,b.ClaimNumber) AS ConversationStartTimestamp
FROM MOLCustomerAgentResponse a
INNER JOIN notesummaries2 b
on a.CaseNumber = b.MOL_Id
;

In [0]:
%sql
create or replace temporary table MemberDetails as
WITH memdetails1 AS (
    select distinct
  a.ClaimNumber,
  g.dateofbirth,
  g.AgeCurrent,
  g.gender,
  g.relationship,
  g.JoinDate,
  g.Postcode,
  g.PremiumAnnualGrossIPT,
  g.PremiumAnnualNetIPT,
  g.HistoryID,
  g.CancellationDate
from
  mol_to_claim a
    Left Join axahealth_dataplatform_pr_gold.base_layer.claim f
      on a.ClaimNumber = f.ClaimReference
    left join axahealth_dataplatform_pr_gold.base_layer.claim_invoice_link cil
      on f.ClaimKey = cil.ClaimKey
      and f.SystemID = cil.SystemID
      and f.memberid = cil.memberid
    left join axahealth_dataplatform_pr_gold.base_layer.members g
      on cil.MemberVersionId = g.MemberVersionId
      and cil.memberid = g.memberid

),
maxmemrow AS (
    SELECT claimnumber
    ,max(HistoryId) AS maxid
    FROM memdetails1
    group by claimnumber
),
memdetails2 AS (
    SELECT distinct b.*
    ,SUBSTRING_INDEX(postcode, ' ', 1) AS PostcodePrefix
    FROM maxmemrow a
    INNER JOIN memdetails1 b
    ON a.claimnumber = b.claimnumber
    AND a.maxid = b.historyid
)

select distinct
  a.ClaimNumber,
  b.dateofbirth,
  b.JoinDate,
  b.cancellationdate,
  b.Relationship,
  b.PremiumAnnualGrossIPT, -- member level from member table -- member paying
  b.PremiumAnnualNetIPT,
  b.AgeCurrent,
  b.Gender,
  b.Postcode,
  b.PostcodePrefix,
  c.UkRegion
from
  mol_to_claim a
    Left Join memdetails2 b
      on a.claimnumber = b.claimnumber
    LEFT JOIN axahealth_dataplatform_pr_gold.base_layer.dim_geography c
    ON b.postcodeprefix = c.PostCode
;


select
  *
from
  MemberDetails;

In [0]:
%sql
Create or replace temporary table ComplainttoContact as
select
  DISTINCT a.*,
  b.casekey,
  b.ComplaintArea
from
  mol_to_claim a
    left join ComplainttoClaim2 b
      on a.ClaimNumber = b.claimNumber;

select
  *
from
  complainttocontact  WHERE casenumber = '11887956' ;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY table complaintRankedResponses2 AS
WITH complaintRankedResponses AS (
  SELECT
    a.*,
    b.ReceiptDate AS ReceiptDateTime,
    DATEDIFF(second, a.conversationstarttimestamp, b.ReceiptDate) AS timediffseconds,
    ROW_NUMBER() OVER (
        PARTITION BY a.casenumber
        ORDER BY
          ABS(DATEDIFF(second, a.conversationstarttimestamp, b.ReceiptDate)),
          b.ReceiptDate, -- Tie-breaker for same time difference
          b.CaseSK -- Additional tie-breaker for deterministic results
      ) AS rn,
    -- Also assign a row number per case key to enforce one case per conversation
    ROW_NUMBER() OVER (
        PARTITION BY b.CaseSK
        ORDER BY ABS(DATEDIFF(second, a.conversationstarttimestamp, b.ReceiptDate))
      ) AS case_rank
  FROM
    ComplainttoContact a
      LEFT JOIN axahealth_dataplatform_pr_silver.respond.dim_case b
        ON a.CaseKey = b.CaseSK
  WHERE
    b.ReceiptDate >= a.conversationstarttimestamp
)
SELECT
  *
FROM
  complaintRankedResponses
WHERE
  rn = 1
  AND case_rank = 1;

SELECT * FROM complaintRankedResponses2 WHERE casenumber = '11887956';

In [0]:
%sql
     CREATE OR REPLACE TEMPORARY VIEW ExGratiaFollowingComplaint AS
     WITH ComplaintDates AS (
           SELECT
        a.claimnumber,
        c.casenumber,
        c.ReceiptDateTime,
        -- Find the first complaint date per claim
        MIN(c.ReceiptDateTime) OVER (PARTITION BY a.claimnumber) AS FirstComplaintDate,
        -- Find the next complaint date after the current complaint
        LEAD(c.ReceiptDateTime) OVER (PARTITION BY a.claimnumber ORDER BY c.ReceiptDateTime) AS NextComplaintDate
    FROM
        mol_to_claim a
    INNER JOIN
        complaintrankedresponses2 c ON a.casenumber = c.casenumber
)
SELECT
    cd.claimnumber,
    cd.casenumber,
    -- Sum ex gratia payments between the first and next complaint dates
    SUM(
        CASE
            WHEN b.PaidDate >= cd.FirstComplaintDate
                 AND (b.PaidDate < cd.NextComplaintDate OR cd.NextComplaintDate IS NULL)
                 AND c.ReceiptDateTime IS NOT NULL
            THEN TRY_CAST(SPLIT_PART(b.TotalExGratiaPaid, '.', 1) AS BIGINT)
            ELSE 0
        END
    ) AS ExGratiaAmountPaid
FROM
    ComplaintDates cd
INNER JOIN
    ExGratiaClaimTotals b ON cd.claimnumber = b.ClaimNumber
LEFT JOIN
    complaintrankedresponses2 c ON cd.casenumber = c.casenumber
GROUP BY
    cd.casenumber, cd.claimnumber;

SELECT * FROM ExGratiaFollowingComplaint;

# Claims - Amount Paid 

In [0]:
%sql
Create or replace temporary view ClaimCost as 
select a.ClaimReference, sum(b.amountclaimed) as TotalClaimed, sum(b.AmountPaid)as ClaimTotalPaid
from axahealth_dataplatform_pr_gold.base_layer.claim a  
left join axahealth_dataplatform_pr_gold.base_layer.invoice b on a.ClaimID = b.ClaimID and a.SystemID = b.SystemID
group by a.ClaimReference;
select * from ClaimCost limit 10 


In [0]:
%sql
create or replace temporary view combined_mol as
select distinct 
  a.*
 ,f.MemberId
  ,b.StandardisedSegment -- current segment
  ,c.PolicySubType -- current policy
  ,f.CurrentCondition
  ,f.CurrentConditionCategory
  ,f.System
  ,CASE WHEN f.CurrentConditionCategory = 'MSK' THEN 1 ELSE 0 END AS MSKClaim
  ,DATEDIFF(day,f.OpenDate, a.CustomerMessageTimestamp) DaysSinceClaimOpened
  ,FLOOR(a.GapFromPreviousMessageSecs/86400.00) AS DaysSincePreviousContact
  ,FLOOR(months_between(f.opendate, g.dateofbirth) / 12) AS ClaimantAge_ClaimOpenDate -- Age when Claim Opened
  , g.AgeCurrent
  , g.JoinDate
  --, DATEDIFF(day, g.JoinDate, GETDATE()) / 365.25 AS Tenure_Years
  ,CASE 
    WHEN g.CancellationDate IS NOT NULL 
        THEN DATEDIFF(day, g.JoinDate, g.CancellationDate) / 365.25
    ELSE DATEDIFF(day, g.JoinDate, GETDATE()) / 365.25
END AS Tenure_Years
  ,g.Relationship
  ,g.PremiumAnnualGrossIPT
  ,g.PremiumAnnualNetIPT
  ,g.Gender
  ,g.CancellationDate 
  ,g.UkRegion
  ,d.ClaimTotalPaid
  ,e.ExGratiaAmountPaid -- exgratia paid following complaint 
  ,h.ComplaintArea
  ,h.ReceiptDateTime as ComplaintReceiptDate
  ,v.VulnerableCustomer
  from
   mol_to_claim a
   Left Join Segmentation b on a.MembershipNumber = b.AXAMembershipNumber
    Left Join MaxPolicySubType c
    on a.ClaimNumber = c.ClaimReference
  Left Join ClaimCost d
    on a.ClaimNumber = d.ClaimReference
    Left Join ExGratiaFollowingComplaint e
      on a.claimnumber = e.claimnumber and a.casenumber = e.casenumber
    Left Join axahealth_dataplatform_pr_gold.base_layer.claim f
      on a.ClaimNumber = f.ClaimReference
    Left Join complaintrankedresponses2 h
    on a.casenumber = h.casenumber 
     left join MemberDetails g on a.ClaimNumber = g.ClaimNumber
     left join VulnerableCustomerClaims v on a.Claimnumber = v.Claimreference
   WHERE f.ClaimReference is not null --remove claims that are not in claims base layer table 
     Order by ClaimNumber, ConversationStartTimestamp asc ;

SELECT * FROM combined_mol WHERE casenumber = '11887956';


In [0]:
%sql
Create or replace temporary view ClaimCountRank as
SELECT
  ContactChannel,
  Casenumber,
  claimnumber,
  COUNT(claimnumber) AS claim_count,
  RANK() OVER (
      PARTITION BY ContactChannel, Casenumber
      ORDER BY COUNT(claimnumber) DESC, claimnumber ASC
    ) AS rnk
FROM
  combined_mol
GROUP BY
  ContactChannel,
  Casenumber,
  claimnumber;

  SELECT * FROM ClaimCountRank;

In [0]:
%sql
Create or replace temporary view ClaimCountRank1 as
select
  *
from
  ClaimCountRank
where
  rnk = 1;

select
  *
from
  ClaimCountRank1

In [0]:
%sql
Create or replace temporary table combined_mol_final as
select distinct
  a.*
from
  combined_mol a
    inner join ClaimCountRank1 b
      on a.casenumber = b.casenumber
      and a.claimnumber = b.claimnumber;

select
  *
from
  combined_mol_final

# Repeat Contact Sequencing

Contact -> Episode -> Sequence -> Customer Journey

**Contact** - Was this interaction Handled?

**Episode** - Did the customer continue contacting within immediate contact window?

**Sequence** - Did the customer return shortly after the episode ended?

**Longer-term repeat** - Did the customer come back later?


Contact	Time	    Gap	Episode	Sequence

C1	    Mon 09:00	—	E1	    S1

C2	    Mon 15:00	6h	E1	    S1

C3	    Tue 10:00	19h	E1	    S1

C4	    Wed 09:00	23h	E1	    S1

C5	    Thu 10:00	25h	E2	    S1

C6	    Fri 09:00	23h	E2	    S1

C7	    Mon 10:00	73h	E3	    S2

With a 24Hr Episode Threshold:

C1 - C4 -> One Episode 

C5 -C6 -> another episode

C7 -> another episode

With 48Hr Threshold:

E1 + E2  - Sequence S1

E3 - Sequence S2

**Pattern A - Immediate Repeat**

Contact -> Contact -> Contact = 1 Episode

This is useful for identifying failure to resolve within the same interaction period.

**Pattern B - Episode -to- episode repeat**

Episode 1 -> 25h -> Episode 2

First episode ended according to your episode definition, but the customer returned very soon afterwards.

Can be evidence for:
- unresolved demand
- delayed resolution
- follow-up requirement
- customer having to contact again
- operation handoffs
- claims/processs dependencies

**Pattern C - Genuine later re-contact**

Episode 1 -> 2 weeks -> Episode 2

This may be a completely different customer need.


## Distribution Analysis to identify Gaps between Contacts for Episode Classification

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW mol_gap_distribution AS 
SELECT Casenumber
,Claimnumber
,previousmessagetimestamp
,CustomerMessageTimestamp
,CustomerMessageSequenceNumberInCase
,GapFromPreviousMessageSecs
,GapFromPreviousMessageSecs/3600.0 AS GapHours
FROM combined_mol_final
WHERE GapFromPreviousMessageSecs IS NOT NULL AND GapFromPreviousMessageSecs >=0;

In [0]:
%sql
SELECT COUNT(*) AS GapCount
    ,ROUND(MIN(GapHours),2) AS MinHours
    ,ROUND(percentile_approx(GapHours,0.01,1000),2) AS P01
    ,ROUND(percentile_approx(GapHours,0.05,1000),2) AS P05
    ,ROUND(percentile_approx(GapHours,0.10,1000),2) AS P10
    ,ROUND(percentile_approx(GapHours,0.25,1000),2) AS P25
    ,ROUND(percentile_approx(GapHours,0.50,1000),2) AS P50
    ,ROUND(percentile_approx(GapHours,0.75,1000),2) AS P75
    ,ROUND(percentile_approx(GapHours,0.90,1000),2) AS P90
    ,ROUND(percentile_approx(GapHours,0.95,1000),2) AS P95
    ,ROUND(percentile_approx(GapHours,0.99,1000),2) AS P99
    ,ROUND(MAX(GapHours),2) AS MaxHours
FROM mol_gap_distribution;

In [0]:
from pyspark.sql import functions as F
mol_gaps = spark.table('mol_gap_distribution')
histogram = mol_gaps.withColumn("GapBucket",F.when(F.col("GapHours") < 1,0).otherwise(F.floor(F.col("GapHours")))).groupBy("GapBucket").count() \
    .withColumnRenamed("count","GapCount") \
    .orderBy("GapBucket")

display(histogram.filter(F.col("GapBucket")<=72))

In [0]:
import matplotlib.pyplot as plt

hist_pd = histogram.filter(F.col("GapBucket")<=72).toPandas()

plt.figure(figsize=(14,6))
plt.bar(
    hist_pd["GapBucket"],
    hist_pd["GapCount"],
    width=0.8
)
plt.axvline(
    24,
    linestyle = "--",
    linewidth=2,
    label="Current Episode = 24h"
)
plt.axvline(
    48,
    linestyle="--",
    linewidth=2,
    label="Current Sequence = 48h"
)

plt.xlabel("Gap from previous customer contact (hours)")
plt.ylabel("Number of Gaps")
plt.title("MOL Contact Gap Distribution")

plt.legend()
plt.grid(axis="y",alpha=0.3)

plt.show()

In [0]:
thresholds = [2,4,6,8,12,16,18,20,24,30,36,48,72]

total_gaps = mol_gaps.count()

results = []

for threshold in thresholds:
    within = mol_gaps.filter(F.col("GapHours")<=threshold).count()
    results.append(
        (
            threshold,
            total_gaps,
            within,
            round(100*within/total_gaps,2)
        )
    )

threshold_df = spark.createDataFrame(results,
                                     [
                                         "ThresholdHours",
                                         "TotalGaps",
                                         "GapsWithinThreshold",
                                         "PercentageWithinThreshold"
                                     ]
                                     )

display(threshold_df.orderBy("ThresholdHours"))

In [0]:
from pyspark.sql.window import Window

thresholds = [6,12,18,20,24,30,36,48,72]

results = []

for threshold in thresholds:
    w = Window.partitionBy("CaseNumber","ClaimNumber").orderBy("CustomerMessageTimestamp","CustomerMessageSequenceNumberInCase")
    df = mol_gaps.withColumn("NewEpisode", F.when(F.col("PreviousMessageTimestamp").isNull() | (F.col("GapHours")>=threshold),1).otherwise(0)) \
                .withColumn("EpisodeId", F.sum("NewEpisode").over(w))
    episode_stats = df.groupBy("Casenumber","ClaimNumber","EpisodeId") \
                        .agg(F.count("*").alias("ContactsPerEpisode"))
    summary = episode_stats.agg(F.count("*").alias("EpisodeCount"), F.avg("ContactsPerEpisode").alias("AvgContactsPerEpisode"),F.expr("percentile_approx(ContactsPerEpisode,0.5)").alias("MedianContactsPerEpisode"),F.sum(F.when(F.col("ContactsPerEpisode") == 1,1).otherwise(0)).alias("SingleContactEpisodes"),F.count("*").alias("TotalEpisodes")).withColumn("SingleContactEpisodePct",F.round(F.col("SingleContactEpisodes")/F.col("TotalEpisodes")*100,2)).withColumn("EpisodeThresholdHours",F.lit(threshold))

    results.append(summary)

episode_sensitivity = results[0]

for r in results[1:]:
    episode_sensitivity = episode_sensitivity.unionByName(r)

display(episode_sensitivity.select("EpisodeThresholdHours","EpisodeCount","AvgContactsPerEpisode","MedianContactsPerEpisode","SingleContactEpisodePct").orderBy("EpisodeThresholdHours"))

In [0]:
threshold_results = []

for threshold in thresholds:
    w = Window.partitionBy("CaseNumber","ClaimNumber").orderBy("CustomerMessageTimestamp","CustomerMessageSequenceNumberInCase")
    df = mol_gaps.withColumn("NewEpisode", F.when(F.col("PreviousMessageTimestamp").isNull() | (F.col("GapHours")>=threshold),1).otherwise(0)) \
                .withColumn("EpisodeId", F.sum("NewEpisode").over(w))
    stats = df.agg(
        F.count("*").alias("TotalContacts"),
        F.sum("NewEpisode").alias("NewEpisodeStarts")
    ).withColumn(
        "EpisodeStartPct",
        F.round(
            F.col("NewEpisodeStarts") /
            F.col("TotalContacts") * 100,
            2
        )
    ).withColumn(
        "EpisodeThresholdHours",
        F.lit(threshold)
    )

    threshold_results.append(stats)

episode_start_sensitivity = threshold_results[0]

for r in threshold_results[1:]:
    episode_start_sensitivity = episode_start_sensitivity.unionByName(r)

display(
    episode_start_sensitivity.select(
        "EpisodeThresholdHours",
        "TotalContacts",
        "NewEpisodeStarts",
        "EpisodeStartPct"
    ).orderBy("EpisodeThresholdHours")
)

In [0]:
final_episode_sensitivity = episode_sensitivity.join(
    episode_start_sensitivity,
    on="EpisodeThresholdHours",
    how="inner"
).select(
    "EpisodeThresholdHours",
    "EpisodeCount",
    "AvgContactsPerEpisode",
    "MedianContactsPerEpisode",
    "SingleContactEpisodePct",
    "NewEpisodeStarts",
    "EpisodeStartPct"
    ).orderBy("EpisodeThresholdHours")

display(final_episode_sensitivity)

In [0]:
import matplotlib.pyplot as plt

pdf = final_episode_sensitivity.toPandas()

fig, ax = plt.subplots(figsize=(10,6))

ax.plot(
    pdf["EpisodeThresholdHours"],
    pdf["EpisodeCount"],
    marker="o",
    label="Episode Count"
)

ax.set_xlabel("Episode Threshold (hours)")
ax.set_ylabel("Number of Episodes")
ax.set_title("MOL Episode Threshold Sensitivity")
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()

In [0]:
import matplotlib.pyplot as plt

pdf = final_episode_sensitivity.toPandas()

fig, ax = plt.subplots(figsize=(10,6))

ax.plot(
    pdf["EpisodeThresholdHours"],
    pdf["SingleContactEpisodePct"],
    marker="o",
    label="Single-Contact Episode %"
)

ax.set_xlabel("Episode Threshold (hours)")
ax.set_ylabel("Percentage")
ax.set_title("MOL Episode Fragmentation by Threshold")
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE ContactEpisodeTable AS
WITH ContactCandidate AS(
    SELECT DISTINCT *
,CASE WHEN GapFromPreviousMessageSecs IS NULL THEN 1
    WHEN GapFromPreviousMessageSecs >= 24 * 60 * 60 THEN 1
    ELSE 0
END AS NewContactCandidate
FROM combined_mol_final
)
SELECT * 
,SUM(NewContactCandidate) OVER(PARTITION BY casenumber,claimnumber ORDER BY CustomerMessageTimestamp, CustomerMessageSequenceNumberInCase ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ContactEpisode
FROM ContactCandidate
ORDER BY Casenumber
;
SELECT * FROM ContactEpisodeTable 
WHERE casenumber = "11887956" ;

## Distribution Analysis to identify Gaps between Contacts for Sequence Classification

In [0]:
contactepisodetable = spark.table("contactepisodetable")

episode_summary = (
    contactepisodetable
    .groupBy(
        "CaseNumber",
        "ClaimNumber",
        "ContactEpisode"
    )
    .agg(
        F.min("CustomerMessageTimestamp").alias("EpisodeStart"),
        F.max("CustomerMessageTimestamp").alias("EpisodeEnd"),
        F.count("*").alias("ContactsInEpisode")
    )
)

display(
    episode_summary
    .orderBy("CaseNumber", "ClaimNumber", "EpisodeStart")
)

In [0]:
episode_window = Window.partitionBy(
    "CaseNumber",
    "ClaimNumber"
).orderBy(
    "EpisodeStart"
)

episode_gaps = (
    episode_summary
    .withColumn(
        "PreviousEpisodeEnd",
        F.lag("EpisodeEnd").over(episode_window)
    )
    .withColumn(
        "GapHoursFromPreviousEpisode",
        (
            F.col("EpisodeStart").cast("long")
            - F.col("PreviousEpisodeEnd").cast("long")
        ) / 3600
    )
)

display(
    episode_gaps
    .select(
        "CaseNumber",
        "ClaimNumber",
        "ContactEpisode",
        "EpisodeStart",
        "EpisodeEnd",
        "PreviousEpisodeEnd",
        "GapHoursFromPreviousEpisode"
    )
    .orderBy("CaseNumber", "ClaimNumber", "EpisodeStart")
)


In [0]:
sequence_gap_distribution = (
    episode_gaps
    .filter(
        F.col("GapHoursFromPreviousEpisode").isNotNull()
    )
    .withColumn(
        "GapBucketHours",
        F.floor("GapHoursFromPreviousEpisode")
    )
    .groupBy("GapBucketHours")
    .count()
    .withColumnRenamed("count", "GapCount")
    .orderBy("GapBucketHours")
)

display(sequence_gap_distribution)


In [0]:
import matplotlib.pyplot as plt

gap_pd = (
    sequence_gap_distribution
    .filter(F.col("GapBucketHours") <= 168)
    .toPandas()
)

plt.figure(figsize=(14, 6))

plt.bar(
    gap_pd["GapBucketHours"],
    gap_pd["GapCount"],
    width=0.8
)

plt.axvline(
    48,
    linestyle="--",
    label="Current Sequence = 48h"
)

plt.xlabel("Gap between Episodes (hours)")
plt.ylabel("Number of Episode Gaps")
plt.title("MOL Inter-Episode Gap Distribution")

plt.legend()
plt.tight_layout()
plt.show()


In [0]:
sequence_thresholds = [
    24,
    36,
    48,
    60,
    72,
    96,
    120,
    168
]

sequence_results = []

for threshold in sequence_thresholds:

    result = (
        episode_gaps
        .filter(
            F.col("GapHoursFromPreviousEpisode").isNotNull()
        )
        .withColumn(
            "NewSequence",
            F.when(
                F.col("GapHoursFromPreviousEpisode") >= threshold,
                1
            ).otherwise(0)
        )
        .agg(
            F.count("*").alias("TotalEpisodeGaps"),
            F.sum("NewSequence").alias("NewSequenceStarts")
        )
        .withColumn(
            "SequenceStartPct",
            F.round(
                F.col("NewSequenceStarts") /
                F.col("TotalEpisodeGaps") * 100,
                2
            )
        )
        .withColumn(
            "SequenceThresholdHours",
            F.lit(threshold)
        )
    )

    sequence_results.append(result)

sequence_sensitivity = sequence_results[0]

for r in sequence_results[1:]:
    sequence_sensitivity = sequence_sensitivity.unionByName(r)

display(
    sequence_sensitivity
    .select(
        "SequenceThresholdHours",
        "TotalEpisodeGaps",
        "NewSequenceStarts",
        "SequenceStartPct"
    )
    .orderBy("SequenceThresholdHours")
)


In [0]:
sequence_sensitivity_results = []

for threshold in sequence_thresholds:

    w = Window.partitionBy(
        "CaseNumber",
        "ClaimNumber"
    ).orderBy("EpisodeStart")

    df = (
        episode_gaps
        .withColumn(
            "NewSequence",
            F.when(
                F.col("PreviousEpisodeEnd").isNull(),
                1
            ).when(
                F.col("GapHoursFromPreviousEpisode") >= threshold,
                1
            ).otherwise(0)
        )
        .withColumn(
            "SequenceNumber",
            F.sum("NewSequence").over(w)
        )
    )

    sequence_summary = (
        df
        .groupBy(
            "CaseNumber",
            "ClaimNumber",
            "SequenceNumber"
        )
        .agg(
            F.count("*").alias("EpisodesInSequence")
        )
    )

    result = (
        sequence_summary
        .agg(
            F.count("*").alias("SequenceCount"),

            F.round(
                F.avg("EpisodesInSequence"),
                3
            ).alias("AvgEpisodesPerSequence"),

            F.expr(
                "percentile_approx(EpisodesInSequence, 0.5)"
            ).alias("MedianEpisodesPerSequence"),

            F.sum(
                F.when(
                    F.col("EpisodesInSequence") == 1,
                    1
                ).otherwise(0)
            ).alias("SingleEpisodeSequences")
        )
        .withColumn(
            "SequenceThresholdHours",
            F.lit(threshold)
        )
        .withColumn(
            "SingleEpisodeSequencePct",
            F.round(
                F.col("SingleEpisodeSequences") /
                F.col("SequenceCount") * 100,
                2
            )
        )
    )

    sequence_sensitivity_results.append(result)

sequence_sensitivity_final = sequence_sensitivity_results[0]

for r in sequence_sensitivity_results[1:]:
    sequence_sensitivity_final = (
        sequence_sensitivity_final.unionByName(r)
    )

display(
    sequence_sensitivity_final
    .select(
        "SequenceThresholdHours",
        "SequenceCount",
        "AvgEpisodesPerSequence",
        "MedianEpisodesPerSequence",
        "SingleEpisodeSequencePct"
    )
    .orderBy("SequenceThresholdHours")
)


In [0]:
seq_pd = (
    sequence_sensitivity_final
    .orderBy("SequenceThresholdHours")
    .toPandas()
)

plt.figure(figsize=(12, 6))

plt.plot(
    seq_pd["SequenceThresholdHours"],
    seq_pd["AvgEpisodesPerSequence"],
    marker="o"
)

plt.axvline(
    48,
    linestyle="--",
    label="48h"
)

plt.xlabel("Sequence Threshold (hours)")
plt.ylabel("Average Episodes per Sequence")
plt.title("MOL Sequence Threshold Sensitivity")

plt.legend()
plt.tight_layout()
plt.show()

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeLevel AS
SELECT ContactChannel
,CaseNumber
,policynumber
,claimnumber
,memberid
,membershipnumber
,ContactEpisode
,MAX(standardisedsegment) AS standardisedsegment
,MAX(policysubtype) AS policysubtype
,MAX(currentcondition) AS currentcondition
,MAX(currentconditioncategory) AS currentconditioncategory
,MAX(system) AS system
,MAX(relationship) AS relationship
,MAX(gender) AS gender
,MAX(ComplaintArea) AS ComplaintArea
,MAX(ComplaintReceiptDate) AS ComplaintReceiptDate
,MAX(VulnerableCustomer) AS VulnerableCustomer
,MAX(UkRegion) AS UkRegion
,MAX(CancellationDate) AS CancellationDate
,MAX(MSKClaim) AS MSKClaim
,MAX(ClaimantAge_ClaimOpenDate) AS ClaimantAge_ClaimOpenDate
,MAX(AgeCurrent) AS AgeCurrent
,MAX(JoinDate) AS JoinDate
,MAX(Tenure_Years) AS Tenure_Years
,ROUND(AVG(DaysSinceClaimOpened)) AS DaysSinceClaimOpened
,COUNT(SecureMessageId) AS MessageCount
,MAX(conversationstarttimestamp) AS conversationstarttimestamp
,MIN(CustomerMessageTimeStamp) AS EpisodeStart
,MAX(CustomerMessageTimestamp) AS EpisodeEnd
,CONCAT_WS('\n', transform(array_sort(collect_list(struct(CustomerMessageTimestamp, CustomerMessageSequenceNumberInCase, CustomerMessage))), x -> x.CustomerMessage)) AS CustomerEpisodeConversation
,CONCAT_WS('\n', transform(array_sort(collect_set(struct(AgentMessageTimestamp, AgentMessageSequenceNumberInCase, AgentMessage,AgentName))), x -> CONCAT(CAST(x.AgentMessageTimestamp AS STRING),' ',COALESCE(x.AgentName, 'Unknown Agent'), ': ', COALESCE(x.AgentMessage,'')))) AS AgentEpisodeConversation
,CONCAT_WS(', ', transform(array_sort(collect_set(struct(AgentMessageTimestamp, AgentMessageSequenceNumberInCase, ResponseAgentId))), x -> x.ResponseAgentId)) AS AgentResponsible
,COUNT(DISTINCT AgentSecureMessageId) AS AgentResponseCount
,CASE WHEN SUM(HandleTimeSecs) > 0 THEN SUM(HandleTimeSecs) ELSE 0 END AS ConversationTimeInSecs
,AVG(PremiumAnnualGrossIPT) AS PremiumAnnualGrossIPT
,AVG(PremiumAnnualNetIPT) AS PremiumAnnualNetIPT
,AVG(ClaimTotalPaid) AS ClaimTotalPaid
,AVG(ExGratiaAmountPaid) AS ExGratiaAmountPaid
FROM ContactEpisodeTable
GROUP BY ALL
ORDER BY CaseNumber,policynumber,claimnumber, ContactEpisode
;
SELECT * FROM EpisodeLevel WHERE casenumber = "11887956";

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeLinks AS
SELECT *
, LAG(ContactEpisode) OVER(PARTITION BY CaseNumber,claimnumber ORDER BY ContactEpisode) AS PreviousContactEpisode
, LAG(EpisodeEnd) OVER(PARTITION BY CaseNumber,claimnumber ORDER BY ContactEpisode) AS PreviousEpisodeEnd
FROM EpisodeLevel

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeLinksWithGap AS
SELECT *
, ROUND((UNIX_TIMESTAMP(EpisodeStart) - UNIX_TIMESTAMP(PreviousEpisodeEnd)) / 3600.0,2) AS GapHours
FROM EpisodeLinks

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeAnalysis AS
SELECT *
, CASE WHEN PreviousContactEpisode IS NULL THEN 0 ELSE 1 END AS HasPreviousEpisode
, CASE WHEN PreviousContactEpisode IS NULL THEN 'FIRST_CONTACT' 
        WHEN GapHours <= 60 THEN 'REPEAT_CANDIDATE'
        ELSE 'LONG_GAP_CANDIDATE'
    END AS RepeatCandidateType
FROM EpisodeLinksWithGap;

SELECT * FROM episodeanalysis WHERE casenumber = "11887956" ORDER BY ContactEpisode;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW ContactSequences AS
WITH SequenceFlags AS (
    SELECT *
    , CASE WHEN PreviousContactEpisode IS NULL THEN 1
        WHEN GapHours > 60 THEN 1 ELSE 0
        END AS NewSequenceFlag
    FROM episodeanalysis
),
SequenceNumbered AS (
    SELECT *
    , SUM(NewSequenceFlag) OVER(PARTITION BY CaseNumber,claimnumber ORDER BY ContactEpisode ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ContactSequence
    FROM sequenceflags
)

SELECT * FROM sequencenumbered;

SELECT * FROM ContactSequences WHERE WHERE casenumber = "11887956" 

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW RepeatContactSequences AS

SELECT CaseNumber,
policynumber,
claimnumber,
        ContactSequence,
        MIN(ContactEpisode) AS SequenceStartEpisode,
        MAX(ContactEpisode) AS SequenceEndEpisode,
        MIN(EpisodeStart) AS SequenceStart,
        MAX(EpisodeEnd) AS SequenceEnd,
        COUNT(*) AS EpisodeCount,
        SUM(MessageCount) AS MessageCount,
        ROUND((UNIX_TIMESTAMP(MAX(EpisodeEnd)) - UNIX_TIMESTAMP(MIN(EpisodeStart)))/3600.0,2) AS SequenceDurationHours
FROM ContactSequences
GROUP BY CaseNumber, ContactSequence,policynumber,claimnumber
;

SELECT * FROM repeatcontactsequences WHERE WHERE casenumber = "11887956" 

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE FinalMOLOutput AS
SELECT cs.contactchannel
,cs.CaseNumber
,cs.policynumber
,cs.claimNumber
,cs.memberid
,cs.membershipnumber
,cs.standardisedsegment
,cs.policysubtype
,cs.currentcondition
,cs.currentconditioncategory
,cs.system
,cs.relationship
,cs.gender
,cs.complaintarea
,cs.complaintreceiptdate
,cs.vulnerablecustomer
,cs.ukregion
,cs.cancellationdate
,cs.mskclaim
,cs.ClaimantAge_ClaimOpendate
,cs.Agecurrent
,cs.joindate
,cs.tenure_years
,cs.dayssinceclaimopened
,cs.ContactEpisode
,cs.ContactSequence
,cs.conversationstarttimestamp
,cs.EpisodeStart
,cs.EpisodeEnd
,cs.PreviousContactEpisode
,cs.PreviousEpisodeEnd
,cs.GapHours
,cs.conversationtimeinsecs
,cs.premiumannualgrossipt
,cs.premiumannualnetipt
,cs.claimtotalpaid
,cs.exgratiaamountpaid
,cs.MessageCount
,cs.AgentResponseCount
,cs.CustomerEpisodeConversation
,cs.AgentEpisodeConversation
,cs.AgentResponsible
,cs.HasPreviousEpisode
,cs.RepeatCandidateType
,CASE WHEN cs.HasPreviousEpisode = 1 THEN TRUE ELSE FALSE END AS IsRepeatContact
,CASE WHEN cs.ContactEpisode > rcs.sequenceStartEpisode THEN TRUE ELSE FALSE END AS IsRepeatWithinSequence
,CASE WHEN cs.ContactEpisode = rcs.SequenceStartEpisode THEN TRUE ELSE FALSE END AS IsSequenceStart
,rcs.SequenceStartEpisode
,rcs.SequenceEndEpisode
,rcs.SequenceEnd
,rcs.EpisodeCount
,rcs.MessageCount AS SequenceMessageCount
,rcs.SequenceDurationHours
FROM ContactSequences cs
LEFT JOIN RepeatContactSequences rcs
ON cs.CaseNumber = rcs.casenumber
AND cs.ContactSequence = rcs.ContactSequence

;

# MOL Clean Content Code 

In [0]:
%python
mol = spark.table("FinalMOLOutput")

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import html, re

@F.udf(StringType())
def clean_conversation_content(x):
    if x is None:
        return None
    
    s = html.unescape(x)
    s = re.sub(r'\|+', '\n', s)
    s = re.sub(r'(?i)<\s*br\s*/?\s*>', '\n', s)
    s = re.sub(r'(?i)</\s*(p|div|li|h[1-6]ul|ol)\s*>', '\n', s)

    #remoev allr emaining html
    s = re.sub(r'<[^>]+>', ' ', s)
    #put each speaker onto a new line
    s = re.sub(
        r'\s*(CUSTOMER|AGENT|ADVISOR|ASSISTANT|ADVISER)\s*:\s*'
        r'(\d{4}-\d{2}-\d{2}\s+\d{2}:d{2}:\d{2})\s*',
        lambda m: f"\n{m.group(1).upper()}: {m.group(2)}\n",
        s,
        flags=re.IGNORECASE
    )
    #clean spacing
    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r' *\n *', '\n', s)
    s = re.sub(r'\n{3,}', '\n\n', s)

    return s.strip()

In [0]:
%python
mol_clean = (
    mol
    .withColumn(
        "CustomerEpisodeConversation",
        clean_conversation_content(F.col("CustomerEpisodeConversation"))
    ).withColumn(
        "AgentEpisodeConversation",
        clean_conversation_content(F.col("AgentEpisodeConversation"))
    )
)
display(
    mol_clean.select(
        "*"
    )
)
# Create temporary view
mol_clean.createOrReplaceTempView("mol_clean_view")


In [0]:
%sql
DROP TABLE IF EXISTS axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_mol_interaction_analytics;
CREATE TABLE axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_mol_interaction_analytics AS
SELECT DISTINCT * FROM mol_clean_view
;

SELECT * 
FROM axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_mol_interaction_analytics
WHERE casenumber = "11887956" 
ORDER BY casenumber,EpisodeStart
;

In [0]:
_sqldf.printSchema()

# AI Analysis - MOL

In [0]:
%pip install azure.keyvault
%restart_python

In [0]:
import asyncio
import json
import time
import random
import requests
import pandas as pd
import numpy as np

from typing import Dict, Any, List, Tuple
from azure.keyvault.secrets import SecretClient
from msal import ConfidentialClientApplication

MOL_FINAL_TABLE = "axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_mol_interaction_analytics".strip()
mol_sdf = spark.table(MOL_FINAL_TABLE)

print(f"MOL rows:{mol_sdf.count():,}")

display(mol_sdf.select("casenumber","CustomerEpisodeConversation","AgentEpisodeConversation").limit(10))

## Model Configuration

In [0]:
def get_uc_secrets_map(connector: str, 
                       vault_url: str, 
                       *secret_keys: str) -> dict[str, str]:
    credential = dbutils.credentials.getServiceCredentialsProvider(connector)
    client = SecretClient(vault_url=vault_url, credential=credential)
    return {k: client.get_secret(k).value for k in secret_keys}

def get_oauth_token(tenant_id: str, 
                    client_id: str, 
                    client_secret: str, 
                    scopes: list[str]) -> str:
    app = ConfidentialClientApplication(client_id, client_credential=client_secret, authority=f"https://login.microsoftonline.com/{tenant_id}")
    result = app.acquire_token_for_client(scopes=scopes)    
    if "access_token" in result:
        return result["access_token"]

In [0]:
MODEL = "gpt-4o-2024-11-20"
MAX_CONCURRENCY = 20
MAX_TOKENS = 800
RETRIES = 3

MIN_CONVERSATION_CHARS = 20
MAX_CONVERSATION_CHARS = 24000

# Service Connector Name
connector = "z-ppp-pr-dbr-keyvaultcredentials-key05"
vault_url = "https://z-ppp-en1-pr-dala-key05.vault.azure.net/"
client_id_key = "modelgateway-healthsgpt-client-id"
client_secret_key = "modelgateway-healthsgpt-client-secret"

# OAuth tenant and scope
tenant_id = "edd791b6-a6e2-450b-9582-5c29c2cc2d25"
scopes = ["https://axapppuk.onmicrosoft.com/modelgateway-api-pr/.default"]

# Model Gateway Base URL
modelgateway_baseurl = "https://proxy.z-ppp-pr-apim01.xpzcloud.com/modelgateway-api-pr/api/"

# Retrieve Secrets via service connector
secrets = get_uc_secrets_map(connector, vault_url, *(client_id_key, client_secret_key))
client_id = secrets[client_id_key]
client_secret = secrets[client_secret_key]


# Mint token via Azure EntraId
token = get_oauth_token(tenant_id, client_id, client_secret, scopes)
print(token)
print("Model Gateway authentication successful.")

## Conversation Preparation

In [0]:
def trim_conversation(text:str, max_chars:int = MAX_CONVERSATION_CHARS) -> str:
    if not text:
        return ""
    
    text = str(text).strip()

    if len(text) <= max_chars:
        return text
    
    first_part = int(max_chars * 0.60)
    last_part = max_chars - first_part

    return (
        text[:first_part]
        + "\n\n[...CONVERSATION TRUNCATED...]\n\n"
        + text[-last_part:]
    )

def build_mol_transcript(customer_conversation, agent_conversation):
    customer = trim_conversation(customer_conversation or "")
    agent = trim_conversation(agent_conversation or "")

    return f"""
    CUSTOMER CONVERSATION
    ---------------------
    {customer}

    AGENT RESPONSE / CONVERSATION
    -----------------------------
    {agent}
    """.strip()



## The JSON Schema

In [0]:
DEMAND_TYPES = [
    "True Failure Demand",
    "Expected Process Demand",
    "External Dependancy Demand",
    "Customer Choice Demand"
]

JSON_SCHEMA = {
    "name": "mol_demand_analysis",
    "schema": {
        "type": "object",
        "properties": {
            "topics":{
                "type": "array",
                "items":{
                    "type":"object",
                    "properties":{
                        "topic": {
                            "type":"string"
                        },
                        "subtopic":{
                            "type":"string"
                        },
                        "demand_type":{
                            "type":"string",
                            "enum": DEMAND_TYPES
                        },
                        "demand_reason":{
                            "type":"string"
                        },
                        "evidence":{
                            "type":"string"
                        },
                        "resolution_status":{
                            "type": "string",
                            "enum":["Resolved", "Partially Resolved", "Unresolved", "Not Applicable"]
                        },
                        "confidence":{
                            "type": "number",
                            "minimum": 0,
                            "maximum": 1
                        }
                    },
                    "required": [
                        "topic",
                        "subtopic",
                        "demand_type",
                        "demand_reason",
                        "evidence",
                        "resolution_status",
                        "confidence"
                    ],
                    "additionalProperties": False
                },
                "minItems": 1
            },
            "overall_demand_type":{
                "type":"string",
                "enum":DEMAND_TYPES
            },
            "overall_resolution":{
                "type":"string",
                "enum":["Resolved", "Partially Resolved", "Unresolved", "Not Applicable"]
            }
        },
        "required":[
            "topics",
            "overall_demand_type",
            "overall_resolution"
        ],
        "additionalProperties": False
    }
}

## MOL-specific prompt

In [0]:
def build_mol_prompt(transcript:str) -> str:
    
    return f"""
    You are an expert customer service and First Contact Resolution (FCR) Analyst working with AXA Health interaction data.

    Analyse the following customer + agent conversation.

    Your task is to identify ALL meaningful customer topics and subtopics discussed in the interaction and classify the type of demand represented by each topic.

    -------------------------------------------------------------------------------------
    DEMAND CLASSIFICATION
    -------------------------------------------------------------------------------------

    1. True Failure Demand

    Use this ONLY when there is evidence that the customer is contacting AXA again because something was not resolved, was incorrectly handled, was missed, or the customer was required to make another contact unnecessarily.

    Examples:
    - "I contacted you last week and this still hasn't been fixed."
    - Customer was promised an action that did not happen.
    - Customer has to contact AXA again because the previous interaction failed to resolve the issue.

    IMPORTANT:
    Do NOT classify something as True Failure Demand merely because the customer is making a repeat contact.

    There must be evidence of a failed or incomplete resolution.

    -------------------------------------------------------------------------------------

    2. Expected Process Demand

    The customer is contacting AXA because a normal part of the service/process requires another interaction.

    Examples:
    - Providing additional information.
    - Checking the progress of a claim.
    - Asking what happens next in a process.
    - Routine documentation or administrative steps.
    - Normal follow-up required by the process.

    This is not necessarily a failure.

    -------------------------------------------------------------------------------------

    3. External Dependency Demand

    The customer needs something that depends on another party or event outside the advisor's direct control.

    Examples:
    - Hospital
    - Consultant
    - GP
    - Provider
    - Third-party organisation
    - Appointment availability
    - Medical Report
    - External authorisation
    - Waiting for another partyto provide information

    The contact may be legitimate even when the customer has contacted AXA previously.

    -------------------------------------------------------------------------------------

    4. Customer Choice Demand

    The customer is making a choice, preference or optional request rather than correcting a service failure.

    Examples:
    - Choosing a provider.
    - Asking for alternative options.
    - Changing an appointment preference.
    - Requesting a different option when the existing option is already available.
    - Voluntary changes initiated by the customer.

    -------------------------------------------------------------------------------------

    IMPORTANT FCR RULE

    Repeat contact does NOT automatically mean Failure Demand.

    The classification must distinguish:

    Repeat Contact
            ↓
    Was the previous issue actually unresolved?
            ↓
    Yes → possible True Failure Demand
    No → leitimate repeat contact

    Therefore, only assign True Failure Demand when the conversation contains evidence supporting the failure.

    -------------------------------------------------------------------------------------
    TOPIC IDENTIFICATION
    -------------------------------------------------------------------------------------

    Identify every meaningful topic discussed.

    For each topic provide:

    - Topic
    - Specific subtopic
    - Demand Type
    - Short explanation for why the demand type applies
    - Supporting evidence from the conversation
    - Resolution status
    - Confidence

    Do not create artificial topics for greetings, pleasantries or irrelavant conversation.

    If several parts of the conversation belon to the same topic, combine them.

    If genuinely different customer intents occur, return multiple topics.

    -------------------------------------------------------------------------------------
    OVERALL DEMAND TYPE
    -------------------------------------------------------------------------------------

    Select the demand type that best represents the main reason for the customer's contact.

    Do not choose True Failure Demand simply because the interaction contains a repeat contact.

    -------------------------------------------------------------------------------------
    CONVERSATION
    ------------------------------------------------------------------------------------- 

    {transcript}

    """.strip()

## Model Call

In [0]:
def post_with_bearer(api_url: str, 
                     token: str, 
                     json_payload: dict, 
                     verify: str | bool = True, 
                     timeout: int = 60) -> requests.Response:
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    resp = requests.post(api_url, headers=headers, json=json_payload, verify=verify, timeout=timeout)
    resp.raise_for_status()
    return resp

def call_mol_model(
    prompt: str,
    max_tokens:int = MAX_TOKENS,
    retries:int = RETRIES
)-> Tuple[Dict[str, Any], Dict[str, Any]]:
    api_url = f"{modelgateway_baseurl}secure-gpt-openai/openai/deployments/{MODEL}/chat/completions?api-version=2024-06-01"


    payload = {
        "messages": [
            {
                "role": "system",
                "content":(
                    "You are an expert AXA Health customer service and FCR analyst."
                )
            },
            {
                "role":"user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "max_tokens": max_tokens,

        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": JSON_SCHEMA["name"],
                "schema": JSON_SCHEMA["schema"],
                "strict": True
            }
        }
    }

    last_error = None

    for attempt in range(retries):

        start = time.time()

        try:
            response = post_with_bearer(
                api_url,
                token,
                payload
            )
            request_id = None

            try:
                request_id = response.headers.get("x-request-id")
            except Exception:
                pass

            latency = time.time() - start

            result = response.json()

            message = result["choices"][0]["message"]

            if "parsed" in message:
                analysis = message["parsed"]
            elif message.get("content"):
                analysis = json.loads(
                    message["content"]
                )
            else:
                raise ValueError("No model content returned")

            usage = result.get("usage",{})
            if not usage:
                raise ValueError(f"No usage returned: {r}")
            
            prompt_tokens = usage.get("pompt_tokens")
            completion_tokens = usage.get("completion_tokens")
            total_tokens = usage.get("total_tokens")

            if prompt_tokens is None:
                if total_tokens is not None and completion_tokens is not None:
                    try:
                        prompt_tokens = int(total_tokens) - int(completion_tokens)
                    except Exception:
                        prompt_tokens = None
            
            meta = {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens,
                "api_latency": latency,
                "request_id": request_id,
                "attempt": attempt + 1
            }

            return analysis, meta

        except Exception as e:
            if attempt == retries - 1:
                raise e
            time.sleep(2 ** attempt)
    
    raise RuntimeError(f"Model call failed after {retries} attempts: {last_error}")

## LLM Analysis Test

In [0]:
from pyspark.sql import functions as F

test_sdf = (
    mol_sdf.select(
        "casenumber"
        ,"contactepisode"
        ,"CustomerEpisodeConversation"
        ,"AgentEpisodeConversation"
    ).filter(F.col("casenumber") == "11887956").orderBy(F.col("ContactEpisode"))
)

test_df = test_sdf.toPandas()

test_results = []

for _,row in test_df.iterrows():

    transcript = build_mol_transcript(
        row["CustomerEpisodeConversation"],
        row["AgentEpisodeConversation"]
    )

    prompt = build_mol_prompt(transcript)

    (analysis, meta) = call_mol_model(prompt)

    test_results.append({
        "casenumber": row["casenumber"]
        ,"contactepisode": row["contactepisode"]
        ,"analysis": analysis
        ,"prompt_tokens": meta.get("prompt_tokens")
        ,"completion_tokens": meta.get("completion_tokens")
        ,"total_tokens": meta.get("total_tokens")
        ,"api_latency": meta.get("api_latency")
        ,"request_id": meta.get("request_id")
        ,"attempt": meta.get("attempt")
    })


test_results_df = pd.DataFrame(test_results)
display(test_results_df)



In [0]:
def flatten_mol_analysis(row):

    analysis = row["analysis"]

    if not analysis:

        return {
            "AI_Topics": None
            ,"AI_Subtopics": None
            ,"AI_DemandTypes": None
            ,"AI_DemandReasons": None
            ,"AI_Evidence": None
            ,"AI_OverallDemandType": None
            ,"AI_OverallResolution": None
        }

    topics = analysis.get("topics",[])
    return {
        "AI_Topics": json.dumps(
            [x.get("topic") for x in topics],
            ensure_ascii= False
        ),
        "AI_Subtopics": json.dumps(
            [x.get("subtopic") for x in topics],
            ensure_ascii=False
        ),
        "AI_DemandTypes": json.dumps(
            [x.get("demand_reason") for x in topics],
            ensure_ascii=False
        ),
        "evidence": json.dumps(
            [x.get("evidence") for x in topics],
            ensure_ascii=False
        ),
        "AI_OverallDemandType": analysis.get("overall_demand_type"),
        "AI_OverallResolution": analysis.get("overall_resolution")
    }

flattened = test_results_df.apply(
    flatten_mol_analysis,
    axis=1,
    result_type="expand"
)

test_output = pd.concat(
    [test_results_df, flattened],
    axis=1
)

display(test_output)